# Celestial Navigation with EqF

## First section - Process star images

Package installations

In [1]:
%pip install sympy matplotlib scipy astropy sep opencv-python pyproj -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# take functions from other hw  ( replace them with SO(3) versions ) 
import numpy as np
def so3_wedge(w):
    w = w.flatten()
    return np.array([
        [0, -w[2], w[1]],
        [w[2], 0, -w[0]],
        [-w[1], w[0], 0]
    ])

def so3_exp(w):
    w = w.flatten()
    theta = np.linalg.norm(w)
    I = np.eye(3)
    
    if theta < 1e-7:
        return I
    
    w_hat = w / theta
    w_wedge = so3_wedge(w_hat)
    
    # Rodrigues' Rotation Formula
    R = I + np.sin(theta) * w_wedge + (1 - np.cos(theta)) * (w_wedge @ w_wedge)
    return R

def so3_vee(Phi):
    # Extracts [w1, w2, w3] from the skew-symmetric form
    return np.array([Phi[2, 1], Phi[0, 2], Phi[1, 0]])

def ad_so3(R):
    # For SO(3), Adjoint(R) is simply R
    return R
    
def so3_log(R):
    I = np.eye(3)
    cos_theta = np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    
    if abs(theta) < 1e-7:
        return np.zeros(3)
    
    # Extract the skew-symmetric part
    w_wedge = (theta / (2 * np.sin(theta))) * (R - R.T)
    return so3_vee(w_wedge)

Plate solve images from the video

In [2]:
import numpy as np
np.math = __import__('math')  # tetra needs np math
import os
import subprocess
import sys
from pathlib import Path
import tetra3
from PIL import Image
import glob

astap_path = r"C:\Program Files\astap\astap.exe"
##imgpath = r"C:\Users\selen\Documents\puthon\590\CNV\CNV\galaxies.png"
fitspath = r"C:\Users\selen\Documents\puthon\590\CNV\CNV\starimg.fits"
starpath_str = r"C:\Users\selen\Documents\puthon\590\CNV\star_images"

# convert png image to fits
def png_to_fits(pngpath, fitspath):
    img = Image.open(pngpath).convert('L')
    # into np array
    data = np.flipud(np.array(img))
    # header data unit
    hdu = fits.PrimaryHDU(data)
    #save to disk
    hdu.writeto(fitspath, overwrite = True)

    return fitspath

# use tetra3 to solve the star image(s) in the directory. taken from tetra3 github example

sys.path.append('..')

#DIR = Path(__file__).parent

# Create instance and load the default database, built for 30 to 10 degree field of view.
# Pass `load_database=None` to not load a database, or to load your own.
t3 = tetra3.Tetra3()

#path = DIR / 
starpath = Path(starpath_str)

for impath in starpath.glob('*'):
    print('Solving for image at: ' + str(impath))
    with Image.open(str(impath)) as img:
        img_gray = img.convert('L') # convert it to grayscale
        # Here you can add e.g. `fov_estimate`/`fov_max_error` to improve speed or a
        # `distortion` range to search (default assumes undistorted image). There
        # are many optional returns, e.g. `return_matches` or `return_visual`. A core
        # aspect of the solution is centroiding (detecting the stars in the image).
        # You can use `return_images` to get a second return value to check the
        # centroiding process, the key `final_centroids` is especially useful.
        solution = t3.solve_from_image(img_gray, distortion=[-.2, .1])

if solution['RA'] is not None:
        ra = solution['RA']
        dec = solution['Dec']
        roll = solution['Roll']
        print(f"RA: {ra}, Dec: {dec}")
else:
        ra = dec = roll = None
        print("Image was not solved")



2026-05-02 15:52:20,508:tetra3.Tetra3-INFO: Loading database from: c:\Users\selen\Documents\puthon\590\.venv\Lib\site-packages\tetra3\data\default_database.npz


NameError: name 'solution' is not defined

Define the EqF class

In [182]:
import numpy as np

class Equivariant:

    def __init__(self, dt, sigma_x, sigma_y, sigma_z, sigma_gravity, Xinit):
        self.dt = dt

        # state (world frame)
        self.xi = np.array([0,0,1]) # state in world frame ?
        self.X = so3_exp(self.xi)

        self.xi0 = np.array([0,0,1]) 
        self.X0 = so3_exp(self.xi0)
    
        # initial condition for estimate
        self.ghat = np.array([0,0,1])
        # process noise covariance
        self.Q = np.eye(3) * sigma_gravity
        # sensor noise covariance
        self.Rcov = np.diag([sigma_x, sigma_y, sigma_z])
        self.B = np.array([[1, 0, 0], 
              [0, 1, 0]])
        self.Xhat = Xinit
        self.Delta = np.zeros(3)
        self.Rlocal = self.B @ self.Rcov @ self.B.T

        # initialize ricatti term
        self.sigma = np.eye(2) *0.1 # represents the initial uncertainty (of calculated initial states)
        self.Cts = np.array([[0,1],[-1,0]])

    # functions for easier computation: 
    def lift(self,in1, in2): # lift S2 element to lie algebra (go from g to xi)
        return so3_exp(in2)  
    def unlift(self,A): # lie algebra to S2 (go from xi to g)
        return so3_log(A)
    
    def phi(self, in1, in2): # left group action
        return in1.T @ in2
    def psi(self, in1, in2): # right group action
        return in1.T @ in2
    
    def left_mult(self, X, A): # left multiplication
        return A@X
    def right_mult(self,X, B): # right multiplication
        return X@B
    
    def right_inverse(self,A):
        return A.T @ np.linalg.inv(A@A.T)
    
    def theta(self,e): # local coordinate chart for S2
        e1 = np.array([0,0,1]) # origin   
        cross = np.cross(e1, e)
        dot = np.dot(e1, e)
        arr = np.array([[0,1,0],[0,0,1]])
        den = np.linalg.norm(cross)
        if den < 1e-12:
            return np.zeros(2)
        
        axis = cross / den
        angle = np.atan2(den, dot)
        t = angle * np.array([axis[0], axis[1]])
        return t
    
    def theta_inv(self, epsilon):
        e1 = np.array([0,0,1]).T
        vec = np.array([epsilon[0], epsilon[1], 0])
        tinv = so3_exp(vec) @ e1
        return tinv

    def predict(self, u, dt): # u is the angular velocity measurement
        # update xhat 
        self.Xhat =  self.Xhat @ so3_exp(u * dt)

        # error dynamics
        A = np.zeros((2,2))
       # B = np.array([[0,1,0],[0,0,1]]) @ R # 2 X 3  #flip??
        self.Rlocal = self.B @ self.Rcov @ self.B.T
        M = self.B @ self.Q @ self.B.T # projected process noise 

        # covariance propagation
        sigma_dot = A @ self.sigma + self.sigma @ A.T + M # 2x2
        self.sigma += sigma_dot * dt
    
    def update(self, g): #g is the gravity measurement 
        # innovation
        e = self.phi(self.Xhat, g)
        innov = self.theta(e) # lical e, epsiloin

        # compute state and output gain matrices
        S = self.Cts @ self.sigma @ self.Cts.T + self.Rlocal
        gain = self.sigma @ self.Cts.T @ np.linalg.inv(S) # 2x2

        delta2d = gain @ innov # 2x1

        # lift to SO(3)
        self.Delta = np.array([delta2d[0], delta2d[1], 0])
        self.Xhat = self.Xhat @ so3_exp(self.Delta) 

        # cov update
        I = np.eye(2)
        self.sigma = (I - gain @ self.Cts) @ self.sigma
        
        # reset correction
        self.Delta = np.zeros(3)
        # Xinv = np.linalg.inv(self.X)
        # # convert into local coordinates
        # local_e = self.theta(e)
        # E = np.linalg.inv(self.Xhat) @ self.X # group error element
        # D = -so3_wedge(self.xi0) # group action derivative
        # self.Delta = np.linalg.pinv(D) @ self.theta_inv(-self.xi0)




Get rotation matrices

In [177]:
import numpy as np
import sympy as sp
from astropy.coordinates import SkyCoord
import astropy.units as units

# create initial rotation based o solved image
# roll is rotation of the camera around its own axis
def initial_rotation(ra, dec, roll):
    # coordinate object for the center of the image
    c = SkyCoord(ra=ra*units.degree,dec = dec*units.degree, frame = 'icrs')

    ra_rad = np.radians(ra)
    dec_rad = np.radians(dec)

    z_b_eci = np.array([
        np.cos(dec_rad)* np.cos(ra_rad),
        np.cos(dec_rad)* np.sin(ra_rad),
        np.sin(dec_rad)
    ])

    # z_b_eci = np.array([
    #     np.cos(dec)* np.cos(ra),
    #     np.cos(dec)* np.sin(ra),
    #     np.sin(dec)
    # ])
    # celestial north pole
    n = np.array([0,0,1])

    # body x axis & avoid pole singularities
    if abs(np.dot(n, z_b_eci)) > 0.99:
        x_b_eci = np.array([1,0,0])
    else:
        x_b_eci = np.cross(n, z_b_eci)
        x_b_eci /= np.linalg.norm(x_b_eci)

    y_b_eci = np.cross(z_b_eci, x_b_eci)
    
    # DCM to map camera to sky frame 
    R_cam_to_eci = np.column_stack((x_b_eci, y_b_eci, z_b_eci))
    
    # if camera is rotated, rotate around the local Z-axis
    if roll != 0:
        gamma = np.radians(roll)
        R_roll = np.array([
            [np.cos(gamma),np.sin(gamma), 0],
            [-np.sin(gamma),  np.cos(gamma), 0],
            [0,              0,             1]
        ])
        R_cam_to_eci = R_cam_to_eci @ R_roll
        
    return R_cam_to_eci

def world_gravity(Xhat, gb, gmst_deg):
    # rotate gravity from body to ECI (stars)
    g_eci = Xhat.T @ gb

    # eart shpin rotation matirx
    theta = np.radians(gmst_deg)
    R_stars_to_ecef = np.array([
        [np.cos(theta),np.sin(theta), 0],
        [-np.sin(theta), np.cos(theta), 0],
        [0, 0, 1]
    ])

    # return gravity in the world frame
    gw = R_stars_to_ecef @ g_eci
    return gw

# now, the gravity estimates are in the world frame
# incorporate the ra, dec coordinates to get a position estimation.

def get_latlon(gw):
    #gw = gw / np.linalg.norm(gw)
    
    gx = gw[0] # assuming gw is a tall matrx. 
    gy = gw[1]
    gz = gw[2]

    # horizontal magnitude
    hz = np.sqrt(gx**2 + gy**2)

    lon = np.degrees(np.atan2(gy, gx))
    lat = np.degrees(np.atan2(gz, hz))

    return lat, lon

Call the EqF and use it on the star data to get lat/lon

In [199]:
import pandas as pd

sx = .1
sy = .1
sz = .1
sg = .01

# FOR NOW - CREATE RA AND DEC VECTORS TO USE (before the plate solver is integrated)
#ra_solved, dec_solved = starcircle(11.6628889, 43.4801667, 10, 100)
#ra = 11.6628889
#dec = 43.4801667

# pick a random star above WL right now
#ra = 162.3937500
#dec = 40.9723333
# pich a random star above Freetown, Sierra Leone right now
#ra = 277.2054167
#dec = 6.5836111

# ra =123.5904167  
# dec = 48.7866944
# other WL stars. 

# test some stars in TR.

# test - use polaris 
#ra = 37.96
#dec = 90 
# 830684503.0 # april 29 in J2000 utc
#time = 827288584.0 # marhc 19 8:52 pm - CANYONLANDS PIC
#time = 830433600.0 # WEST LAFAYETTE MAY 2
#time = 831014544649/1000.0 # SL time

#ankara sample
time = 1777766321.0 - 946684800.0
ra = 39.0
dec = 16.0

gravity_path = r"C:\Users\selen\Documents\puthon\590\CNV\CNV\CNV_csvs\Gravity.csv"
gyro_path = r"C:\Users\selen\Documents\puthon\590\CNV\CNV\CNV_csvs\Gyroscope.csv"

gr = pd.read_csv(gravity_path)
gy = pd.read_csv(gyro_path)

gravitydata = gr.values
gyrodata = gy.values

gravtimes = gravitydata[:,0]
gyrotimes = gyrodata[:,0]

# TIMESERIES DATASETS
# sort out the gravity data:
gz = gravitydata[:, 2]
gy = gravitydata[:, 3]
gx = gravitydata[:, 4]
g_measurement = np.vstack((gx, gy, gz)) # gmeas is a long matrix
# gyroscope ANGULAR VLOECITY  data
gyz = gyrodata[:, 2]
gyy = gyrodata[:, 3]
gyx = gyrodata[:, 4]
gyro_data =  np.vstack((gyx, gyy, gyz)) # a long matrix 

steps = len(g_measurement[1,:])

X0 = initial_rotation(ra, dec,0)

Xhat_storage = [X0.copy()]
dt = np.abs(gravtimes[0]-gravtimes[1])
dt2 = np.abs(gyrotimes[0] - gyrotimes[1])

eqf = Equivariant(dt2, sigma_x=sx, sigma_y=sy, sigma_z = sz, sigma_gravity = sg, Xinit=X0)

gmst_deg = .004167 * (24110.54841 + 8640184.812866 * time + .093104*(time**2) - (6.2e-6)*(time**3) )
gmst_deg = gmst_deg % 360
gw = np.zeros((steps, 3)) # initialize gw as a tall matrix 

# filtering loop
for k in range(steps):
    g = g_measurement[:, k]
    omega = gyro_data[:, k]

    g = g / np.linalg.norm(g)
    eqf.predict(omega, dt2)
    eqf.update(g)
    
    Xhat_storage.append(eqf.Xhat)
    gmst_deg = (gmst_deg + (15.041/3600.0) * (k * dt2)) % 360 # account for earth rotation

    gw[k,:] = world_gravity(eqf.Xhat, g, gmst_deg) 

gwmean_x = np.mean(gw[:, 0])
gwmean_y = np.mean(gw[:, 1])
gwmean_z = np.mean(gw[:, 2])
gwmean_final = np.array([gwmean_x, gwmean_y, gwmean_z])  # average the world frame gravity vectors 

lat, lon = get_latlon(gwmean_final)
lat /= 2
lon /=2

dir_lat = ""
dir_long = ""

if lon > 0:
    dir_long = "West"
else:
    dir_long = "East"
    lon = np.abs(lon)

if lat > 0:
    dir_lat = "North"
else:
    dir_lat = "South"
    lat = np.abs(lat)

print("Latitude: ", lat,dir_lat)
print("Longitude: ", lon, dir_long)


# lat lon of canyonlands (reference pic location): 38.21 N 109.90 W
# west lafaette: lat 40.425869, lon -86.90806
#@ self.X0.T

#ankara; 39.9334° N, 32.8597° E

# 69 mi off

Latitude:  40.79455653162196 North
Longitude:  32.18329549670111 East
